<a href="https://colab.research.google.com/github/dhiru69-tech/ML-Techniques-Lab-Project/blob/main/notebooks/day10_decision_tree_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 10: Decision Tree, Pruning and Tuning
**Course:** BO CDA 205, Machine Learning Techniques
**Dataset:** German Credit (1000 customers, 21 columns)
**Syllabus link:** Supervised learning, Decision Trees (controlling overfitting)

> **Goal of today:** Prune the tree, choose its settings with cross-validation on the training set only, and evaluate the final tree on the test set once.

## 1. Setup

Same workflow as before. New today: a **Pipeline** that bundles preprocessing and the tree. During cross-validation, the preprocessor is then fitted again inside each fold using only that fold's training rows, so no information leaks.

`make_pipe(**params)` builds a fresh pipeline with the tree settings we pass in.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix)

# Load data. bad = 1 is the positive class
data = fetch_openml(name="credit-g", version=1, as_frame=True)
df = data.frame
X = df.drop(columns=['class'])
y = df['class'].map({'good': 0, 'bad': 1}).astype(int)

# Stratified 80/20 split with a fixed seed
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

num_cols = X.select_dtypes(include='number').columns.tolist()
cat_cols = X.select_dtypes(exclude='number').columns.tolist()

preprocess = ColumnTransformer(
    [('num', StandardScaler(), num_cols),
     ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)],
    sparse_threshold=0
)
preprocess.set_output(transform="pandas")


def make_pipe(**params):
    """Pipeline = preprocessing + decision tree. params go to the tree."""
    return Pipeline([
        ('prep', clone(preprocess)),
        ('tree', DecisionTreeClassifier(random_state=42, **params)),
    ])


def evaluate(name, y_true, pred):
    """Return the main metrics for the positive class (bad = 1)."""
    tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()
    return {
        'model': name,
        'accuracy': round(accuracy_score(y_true, pred), 3),
        'precision': round(precision_score(y_true, pred, zero_division=0), 3),
        'recall': round(recall_score(y_true, pred, zero_division=0), 3),
        'f1': round(f1_score(y_true, pred, zero_division=0), 3),
        'FN': fn, 'FP': fp,
        'cost': 5 * fn + fp,          # FN costs 5, FP costs 1 (Day 8)
    }


# 5-fold cross-validation, keeping the 70/30 ratio inside every fold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print("Train:", X_train.shape, "| Test:", X_test.shape)